In [41]:
import pandas as pd
import numpy as np

In [42]:
am_ground = pd.read_csv("C:/Users/HELIOS-300/Desktop/WAVES/AM Full Code/Cameron_AM_Clean.csv")
am_ground.head()

,id,do_session,date_time,time,time_relative_new,Modifier_1,Modifier_2,intensity_do,Comment,Activity_Type,posture_wbm,broad_domain,broad.behavior_do,posture_broad,broad.posture_do
0,1.0,DO1,2017-10-06 16:43:57,16:43:57,00:00:00,No movement,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary
1,1.0,DO1,2017-10-06 16:43:58,16:43:58,00:00:01,No movement,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary
2,1.0,DO1,2017-10-06 16:43:59,16:43:59,00:00:02,No movement,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary
3,1.0,DO1,2017-10-06 16:44:00,16:44:00,00:00:03,No movement,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary
4,1.0,DO1,2017-10-06 16:44:01,16:44:01,00:00:04,No movement,NaN,NaN,NaN,les_social,stand,leisure,leisure_inactive,stand_move,stationary


In [43]:
am_gt = pd.read_csv("C:/Users/HELIOS-300/Desktop/Data/am_gt_3.csv")
am_gt.head()

,day,id,actual_time,time,coding,primary_behavior,primary_posture,primary_upperbody,primary_intensity,num_postures,transition,posture_coding,broad_activity,detailed_activity,updated_activity,DO_session
0,7/24/2017,AM02,13:17:10,13H 17M 10S,non-sed,HA- housework,private/not coded,unknown,private/not coded,1,0,private/not coded,private/not coded,private/not coded,private/not coded,DO1
1,7/24/2017,AM02,13:17:11,13H 17M 11S,non-sed,HA- housework,private/not coded,unknown,private/not coded,1,0,private/not coded,private/not coded,private/not coded,private/not coded,DO1
2,7/24/2017,AM02,13:17:12,13H 17M 12S,non-sed,HA- housework,private/not coded,unknown,private/not coded,1,0,private/not coded,private/not coded,private/not coded,private/not coded,DO1
3,7/24/2017,AM02,13:17:13,13H 17M 13S,non-sed,HA- housework,private/not coded,unknown,private/not coded,1,0,private/not coded,private/not coded,private/not coded,private/not coded,DO1
4,7/24/2017,AM02,13:17:14,13H 17M 14S,non-sed,HA- housework,LA- stand and move with upper body movement,unknown,light,2,1,LA- stand and move light,mixed-activity,housework,mixed-activity,DO1


In [44]:
print("AM Groundtruth ID Unique:\n", am_ground["id"].unique(), "\n\nAM Groundtruth DO Unique:\n", am_ground["do_session"].unique())

AM Groundtruth ID Unique:
 [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. 20. 21. 22. 24. 25. 26. 27.] 

AM Groundtruth DO Unique:
 ['DO1' 'DO2' 'DO2_a' 'DO2_b' 'DO1_a' 'DO1_b']


In [45]:
print("AM GT ID Unique:\n", am_gt["id"].unique(), "\n\nAM GT DO Unique:\n", am_gt["DO_session"].unique())

AM GT ID Unique:
 ['AM02' 'AM03' 'AM06' 'AM07' 'AM08' 'AM09' 'AM10' 'AM11' 'AM04' 'AM12'
 'AM05' 'AM13' 'AM14' 'AM01' 'AM16' 'AM15' 'AM17' 'AM18' 'AM19' 'AM20'
 'AM21' 'AM22' 'AM24' 'AM25' 'AM26' 'AM27'] 

AM GT DO Unique:
 ['DO1' 'DO2']


In [46]:
print("AM Ground Shape: ", am_ground.shape)
print("\nAM GT Shape: ", am_gt.shape)

AM Ground Shape:  (397530, 15)

AM GT Shape:  (332985, 16)


In [47]:
# Pair-level summary: duration, rows, gaps, duplicates, and mismatches

def normalize_am_ground_id(series):
    # map 1.0 -> AM01, 2.0 -> AM02, etc.
    return (
        series.dropna().astype(int).astype(str).str.zfill(2).radd("AM")
    )

am_ground_norm = am_ground.copy()
am_ground_norm["id_norm"] = pd.NA
am_ground_norm.loc[am_ground_norm["id"].notna(), "id_norm"] = normalize_am_ground_id(am_ground_norm["id"])

# align session column name
am_ground_norm = am_ground_norm.rename(columns={"do_session": "DO_session"})

# parse time columns
am_ground_norm["time_norm"] = pd.to_datetime(am_ground_norm["time"], format="%H:%M:%S", errors="coerce")
am_ground_norm["datetime_norm"] = pd.to_datetime(am_ground_norm["date_time"], errors="coerce")

am_gt_norm = am_gt.copy()
am_gt_norm["time_norm"] = pd.to_datetime(am_gt_norm["actual_time"], format="%H:%M:%S", errors="coerce")
am_gt_norm["datetime_norm"] = pd.to_datetime(
    am_gt_norm["day"].astype(str).str.strip() + " " + am_gt_norm["actual_time"].astype(str).str.strip(),
    errors="coerce",
)


def summarize_pairs(df, id_col, session_col, time_col, datetime_col=None):
    cols = [id_col, session_col, time_col]
    if datetime_col is not None and datetime_col in df.columns:
        cols.append(datetime_col)

    use = df[cols].dropna(subset=[id_col, session_col, time_col]).copy()
    use = use.sort_values([id_col, session_col, time_col])

    grouped = use.groupby([id_col, session_col])
    summary = grouped[time_col].agg(
        first_time="min",
        last_time="max",
        row_count="size",
        unique_time_count="nunique",
    ).reset_index()

    summary["duration_seconds"] = (
        (summary["last_time"] - summary["first_time"]).dt.total_seconds().astype("Int64")
    )
    summary["duration_hms"] = (
        pd.to_timedelta(summary["duration_seconds"], unit="s")
        .astype(str)
        .str.replace("0 days ", "", regex=False)
    )
    summary["expected_count"] = summary["duration_seconds"] + 1
    summary["gap_seconds"] = summary["expected_count"] - summary["unique_time_count"]

    # True duplicate rows should use full datetime when available.
    if datetime_col is not None and datetime_col in use.columns:
        unique_row_count = (
            use.dropna(subset=[datetime_col])
            .drop_duplicates([id_col, session_col, datetime_col])
            .groupby([id_col, session_col], sort=False)
            .size()
            .rename("unique_row_count")
            .reset_index()
        )
        summary = summary.merge(unique_row_count, on=[id_col, session_col], how="left")
        summary["unique_row_count"] = summary["unique_row_count"].fillna(0).astype("Int64")
        summary["duplicate_rows"] = summary["row_count"] - summary["unique_row_count"]
        summary = summary.drop(columns=["unique_row_count"])
    else:
        summary["duplicate_rows"] = summary["row_count"] - summary["unique_time_count"]

    summary["first_time"] = summary["first_time"].dt.time
    summary["last_time"] = summary["last_time"].dt.time

    # count number of gap breaks (> 1 sec) using unique timestamps
    use_unique = use.drop_duplicates([id_col, session_col, time_col])
    diffs = use_unique.groupby([id_col, session_col])[time_col].diff().dt.total_seconds()
    gap_count = (
        diffs.gt(1)
        .groupby([use_unique[id_col], use_unique[session_col]])
        .sum()
        .rename("gap_count")
        .reset_index()
    )

    summary = summary.merge(
        gap_count,
        on=[id_col, session_col],
        how="left",
    )
    if "gap_count" not in summary.columns:
        summary["gap_count"] = 0
    summary["gap_count"] = summary["gap_count"].fillna(0).astype(int)

    # standardize id/session column names for downstream usage
    summary = summary.rename(columns={id_col: "id", session_col: "DO_session"})

    return summary.sort_values(["id", "DO_session"]).reset_index(drop=True)


am_ground_summary = summarize_pairs(
    am_ground_norm,
    id_col="id_norm",
    session_col="DO_session",
    time_col="time_norm",
    datetime_col="datetime_norm",
)

am_gt_summary = summarize_pairs(
    am_gt_norm,
    id_col="id",
    session_col="DO_session",
    time_col="time_norm",
    datetime_col="datetime_norm",
)

print("am_ground_summary (duration, rows, gaps, duplicates):")
print(am_ground_summary.to_string(index=False), "\n")
print("am_ground total rows across unique pairs:", am_ground_summary["row_count"].sum(), "\n")

print("am_gt_summary (duration, rows, gaps, duplicates):")
print(am_gt_summary.to_string(index=False), "\n")
print("am_gt total rows across unique pairs:", am_gt_summary["row_count"].sum(), "\n")

# pairs present in one dataset but not the other
pairs_ground = set(zip(am_ground_summary["id"], am_ground_summary["DO_session"]))
pairs_gt = set(zip(am_gt_summary["id"], am_gt_summary["DO_session"]))

only_in_ground = sorted(pairs_ground - pairs_gt)
only_in_gt = sorted(pairs_gt - pairs_ground)

print("Pairs only in am_ground:")
print(only_in_ground, "\n")
print("am_ground rows for pairs only in am_ground:",
      am_ground_summary[
          am_ground_summary[["id", "DO_session"]].apply(tuple, axis=1).isin(only_in_ground)
      ]["row_count"].sum(),
      "\n")

print("Pairs only in am_gt:")
print(only_in_gt)

am_ground_summary (duration, rows, gaps, duplicates):
  id DO_session first_time last_time  row_count  unique_time_count  duration_seconds duration_hms  expected_count  gap_seconds  duplicate_rows  gap_count
AM01        DO1   16:43:57  18:46:13       7337               7337              7336     02:02:16            7337            0               0          0
AM01        DO2   18:44:45  20:45:48       7264               7264              7263     02:01:03            7264            0               0          0
AM02        DO1   13:17:10  15:17:32       7223               7223              7222     02:00:22            7223            0               0          0
AM02      DO2_a   08:00:27  08:52:32       3126               3126              3125     00:52:05            3126            0               0          0
AM02      DO2_b   08:03:56  10:12:29       7715               7714              7713     02:08:33            7714            0               0          0
AM03        DO1   14:0

In [49]:
# QA check: did activity/posture split carry through correctly?

qa = am_ground.copy()

required_cols = ["id", "do_session", "date_time", "Activity_Type", "posture_wbm"]
missing_cols = [c for c in required_cols if c not in qa.columns]
if missing_cols:
    print("Missing required columns:", missing_cols)
else:
    # Normalize empties
    for c in ["Activity_Type", "posture_wbm"]:
        qa[c] = qa[c].astype("string").str.strip()
        qa[c] = qa[c].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

    qa["has_activity"] = qa["Activity_Type"].notna()
    qa["has_posture"] = qa["posture_wbm"].notna()

    total = len(qa)
    both = int((qa["has_activity"] & qa["has_posture"]).sum())
    only_activity = int((qa["has_activity"] & ~qa["has_posture"]).sum())
    only_posture = int((~qa["has_activity"] & qa["has_posture"]).sum())
    neither = int((~qa["has_activity"] & ~qa["has_posture"]).sum())

    print("=== Activity/Posture Split QA (am_ground) ===")
    print(f"Total rows: {total:,}")
    print(f"Both present: {both:,} ({both/total:.2%})")
    print(f"Only activity present: {only_activity:,} ({only_activity/total:.2%})")
    print(f"Only posture present: {only_posture:,} ({only_posture/total:.2%})")
    print(f"Neither present: {neither:,} ({neither/total:.2%})")

    # Session-level missingness hotspots
    by_pair = (
        qa.groupby(["id", "do_session"], dropna=False)
        .agg(
            rows=("id", "size"),
            both=("has_activity", lambda s: 0),  # placeholder, overwritten below
            only_activity=("has_activity", lambda s: 0),
            only_posture=("has_activity", lambda s: 0),
            neither=("has_activity", lambda s: 0),
        )
        .reset_index()
    )

    pair_counts = (
        qa.assign(
            _both=qa["has_activity"] & qa["has_posture"],
            _only_activity=qa["has_activity"] & ~qa["has_posture"],
            _only_posture=~qa["has_activity"] & qa["has_posture"],
            _neither=~qa["has_activity"] & ~qa["has_posture"],
        )
        .groupby(["id", "do_session"], dropna=False)[["_both", "_only_activity", "_only_posture", "_neither"]]
        .sum()
        .reset_index()
        .rename(columns={
            "_both": "both",
            "_only_activity": "only_activity",
            "_only_posture": "only_posture",
            "_neither": "neither",
        })
    )

    by_pair = by_pair.drop(columns=["both", "only_activity", "only_posture", "neither"]).merge(
        pair_counts, on=["id", "do_session"], how="left"
    )

    by_pair["pct_only_activity"] = by_pair["only_activity"] / by_pair["rows"]
    by_pair["pct_only_posture"] = by_pair["only_posture"] / by_pair["rows"]
    by_pair["pct_neither"] = by_pair["neither"] / by_pair["rows"]

    print("\nTop 12 session groups by missing split info (only_activity + only_posture + neither):")
    tmp = by_pair.copy()
    tmp["missing_total"] = tmp["only_activity"] + tmp["only_posture"] + tmp["neither"]
    print(
        tmp.sort_values(["missing_total", "rows"], ascending=[False, False])[
            [
                "id", "do_session", "rows", "both", "only_activity", "only_posture", "neither",
                "pct_only_activity", "pct_only_posture", "pct_neither"
            ]
        ]
        .head(12)
        .to_string(index=False)
    )

    if neither > 0:
        print("\nSample rows where both Activity_Type and posture_wbm are missing:")
        sample_cols = [c for c in ["id", "do_session", "date_time", "time", "time_relative_new", "Activity_Type", "posture_wbm", "Comment"] if c in qa.columns]
        print(
            qa.loc[~qa["has_activity"] & ~qa["has_posture"], sample_cols]
            .head(20)
            .to_string(index=False)
        )


=== Activity/Posture Split QA (am_ground) ===
Total rows: 397,530
Both present: 395,158 (99.40%)
Only activity present: 34 (0.01%)
Only posture present: 2,338 (0.59%)
Neither present: 0 (0.00%)

Top 12 session groups by missing split info (only_activity + only_posture + neither):
  id do_session  rows  both  only_activity  only_posture  neither  pct_only_activity  pct_only_posture  pct_neither
11.0      DO1_b  9431  7093              0          2338        0           0.000000          0.247906          0.0
26.0        DO2 13489 13466             23             0        0           0.001705          0.000000          0.0
 2.0      DO2_b  7715  7704             11             0        0           0.001426          0.000000          0.0
11.0        DO2 14038 14038              0             0        0           0.000000          0.000000          0.0
16.0        DO2  7682  7682              0             0        0           0.000000          0.000000          0.0
14.0        DO1  7679  